# Solutions — Wanderbricks Platform Intelligence (Hard 05)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

bookings    = spark.table("samples.wanderbricks.bookings")
users       = spark.table("samples.wanderbricks.users")
properties  = spark.table("samples.wanderbricks.properties")
reviews     = spark.table("samples.wanderbricks.reviews")
payments    = spark.table("samples.wanderbricks.payments")
hosts       = spark.table("samples.wanderbricks.hosts")
clickstream = spark.table("samples.wanderbricks.clickstream")


## Solution 1 — Host Performance Dashboard

In [ ]:
meta_schema = T.StructType([
    T.StructField("device",   T.StringType()),
    T.StructField("referrer", T.StringType()),
])

host_props = (
    properties
    .groupBy("host_id")
    .agg(F.count("*").alias("total_properties"))
)

prop_ratings = (
    reviews
    .filter(F.col("is_deleted") == False)
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(F.round(F.avg("rating"), 2).alias("avg_property_rating"))
)

host_bookings = (
    bookings
    .filter(F.col("status") == "confirmed")
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(
        F.count("*").alias("total_bookings"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    )
)

result_1 = (
    hosts
    .join(host_props,    on="host_id", how="left")
    .join(prop_ratings,  on="host_id", how="left")
    .join(host_bookings, on="host_id", how="left")
    .select("host_id", "name", "country", "is_verified",
            "total_properties", "total_bookings", "total_revenue", "avg_property_rating")
    .orderBy(F.col("total_revenue").desc())
)


## Solution 2 — Property Occupancy Rate

In [ ]:
result_2 = (
    bookings
    .filter(F.col("status") == "confirmed")
    .withColumn("booked_nights", F.datediff("check_out", "check_in"))
    .groupBy("property_id")
    .agg(F.sum("booked_nights").alias("total_booked_nights"))
    .withColumn("occupancy_rate_pct", F.round(F.col("total_booked_nights") / 365 * 100, 2))
    .join(properties.select("property_id", "title", "property_type"), on="property_id")
    .select("property_id", "title", "property_type", "total_booked_nights", "occupancy_rate_pct")
    .orderBy(F.col("occupancy_rate_pct").desc())
)


## Solution 3 — Clickstream Device Analysis

In [ ]:
meta_schema = T.StructType([
    T.StructField("device",   T.StringType()),
    T.StructField("referrer", T.StringType()),
])

result_3 = (
    clickstream
    .withColumn("meta", F.from_json("metadata", meta_schema))
    .groupBy(
        F.col("meta.device").alias("device"),
        F.col("event"),
    )
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct("user_id").alias("unique_users"),
    )
    .orderBy(F.col("event_count").desc())
)


## Solution 4 — Booking Cancellation Rate by Property Type and User Country

In [ ]:
result_4 = (
    bookings
    .join(users.select("user_id", F.col("country").alias("user_country")), on="user_id")
    .join(properties.select("property_id", "property_type"), on="property_id")
    .groupBy("property_type", "user_country")
    .agg(
        F.count("*").alias("total_bookings"),
        F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_bookings"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(F.col("cancelled_bookings") / F.col("total_bookings") * 100, 2)
    )
    .orderBy(F.col("cancellation_rate_pct").desc())
)


## Solution 5 — Revenue per Available Night (RevPAN)

In [ ]:
total_rev = (
    bookings
    .filter(F.col("status") == "confirmed")
    .groupBy("property_id")
    .agg(F.round(F.sum("total_amount"), 2).alias("actual_revenue"))
)

result_5 = (
    properties
    .join(total_rev, on="property_id", how="left")
    .withColumn("revpan", F.round(F.coalesce(F.col("actual_revenue"), F.lit(0)) / F.col("base_price"), 2))
    .select("property_id", "title", "property_type", "base_price", "actual_revenue", "revpan")
    .orderBy(F.col("revpan").desc())
)


## Solution 6 — User Booking Cohort: First-Booking Month Retention

In [ ]:
first_booking = (
    bookings
    .filter(F.col("status") == "confirmed")
    .groupBy("user_id")
    .agg(F.min("check_in").alias("first_checkin"))
    .withColumn("cohort_month", F.date_trunc("month", "first_checkin"))
)

cohort_size = (
    first_booking
    .groupBy("cohort_month")
    .agg(F.countDistinct("user_id").alias("cohort_size"))
)

m1 = (
    bookings.filter(F.col("status") == "confirmed")
    .join(first_booking, on="user_id")
    .filter(
        (F.months_between("check_in", "cohort_month") >= 1) &
        (F.months_between("check_in", "cohort_month") < 2)
    )
    .groupBy("cohort_month")
    .agg(F.countDistinct("user_id").alias("retained_m1"))
)

m3 = (
    bookings.filter(F.col("status") == "confirmed")
    .join(first_booking, on="user_id")
    .filter(
        (F.months_between("check_in", "cohort_month") >= 3) &
        (F.months_between("check_in", "cohort_month") < 4)
    )
    .groupBy("cohort_month")
    .agg(F.countDistinct("user_id").alias("retained_m3"))
)

result_6 = (
    cohort_size
    .join(m1, on="cohort_month", how="left")
    .join(m3, on="cohort_month", how="left")
    .fillna(0, subset=["retained_m1", "retained_m3"])
    .orderBy("cohort_month")
)


## Solution 7 — Property Price Tier vs Average Rating

In [ ]:
result_7 = (
    properties
    .withColumn("price_tier",
        F.when(F.col("base_price") < 50,   "Budget (<$50)")
         .when(F.col("base_price") < 150,  "Mid ($50-150)")
         .when(F.col("base_price") < 300,  "Premium ($150-300)")
         .otherwise("Luxury ($300+)")
    )
    .join(
        reviews.filter(F.col("is_deleted") == False)
               .groupBy("property_id")
               .agg(F.round(F.avg("rating"), 2).alias("avg_r"),
                    F.count("*").alias("total_reviews")),
        on="property_id", how="left"
    )
    .groupBy("price_tier")
    .agg(
        F.count("*").alias("property_count"),
        F.round(F.avg("avg_r"), 2).alias("avg_rating"),
        F.sum("total_reviews").alias("total_reviews"),
    )
    .orderBy("price_tier")
)


## Solution 8 — Payment Reconciliation

In [ ]:
total_paid = (
    payments
    .filter(F.col("status") == "completed")
    .groupBy("booking_id")
    .agg(F.round(F.sum("amount"), 2).alias("total_paid"))
)

result_8 = (
    bookings
    .join(total_paid, on="booking_id", how="left")
    .withColumn("total_paid",  F.coalesce(F.col("total_paid"), F.lit(0.0)))
    .withColumn("booking_amount", F.col("total_amount"))
    .withColumn("difference",  F.round(F.col("total_paid") - F.col("booking_amount"), 2))
    .withColumn("is_reconciled",
        F.abs(F.col("difference")) / F.col("booking_amount") * 100 <= F.lit(1.0)
    )
    .select("booking_id", "booking_amount", "total_paid", "difference", "is_reconciled")
    .orderBy(F.col("difference").desc())
)
